In [1]:
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

c:\Users\Lil\miniconda3\envs\aca\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
citibike_df = pd.read_csv("../data/citibike/JC/JC2025.csv")

Missing Values

In [4]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,4397,0.004385
10,end_lat,3444,0.003435
11,end_lng,3444,0.003435
6,end_station_name,3235,0.003226
4,start_station_name,3,0.000003
5,start_station_id,3,0.000003
8,start_lat,2,0.000002
9,start_lng,2,0.000002
0,ride_id,0,0.000000
3,ended_at,0,0.000000


In [5]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1002704 entries, 0 to 1002703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   ride_id             1002704 non-null  str    
 1   rideable_type       1002704 non-null  str    
 2   started_at          1002704 non-null  str    
 3   ended_at            1002704 non-null  str    
 4   start_station_name  1002701 non-null  str    
 5   start_station_id    1002701 non-null  str    
 6   end_station_name    999469 non-null   str    
 7   end_station_id      998307 non-null   str    
 8   start_lat           1002702 non-null  float64
 9   start_lng           1002702 non-null  float64
 10  end_lat             999260 non-null   float64
 11  end_lng             999260 non-null   float64
 12  member_casual       1002704 non-null  str    
dtypes: float64(4), str(9)
memory usage: 222.6 MB


Duration

In [6]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'],errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'],errors="coerce")

In [7]:
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [10]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

In [12]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [13]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [14]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-01-16 17:50:49.136,2025-01-16,2025-01,January,Thursday,17,Winter
1,2025-01-31 06:10:41.818,2025-01-31,2025-01,January,Friday,6,Winter
2,2025-01-09 16:42:50.213,2025-01-09,2025-01,January,Thursday,16,Winter
3,2025-01-21 16:14:14.398,2025-01-21,2025-01,January,Tuesday,16,Winter
4,2025-01-30 16:38:18.840,2025-01-30,2025-01,January,Thursday,16,Winter


Storing Enriched Data

In [15]:
citibike_df.to_csv("../data/citibike/JC/JC2025_Enriched.csv", index = False)